# ObjectGraph-Agent pilot on Google Colab (Python projects)
Runs stocktrends, lice and TextCNN with Claude. Your laptop only needs a browser. C++/Java projects need Docker, so run those in GitHub Codespaces (see RUN_PILOT.md).

In [ ]:
# 1. Get the code straight from GitHub (same code reviewers will see)
!git clone -q https://github.com/wijayasekarageyashmithadilshan-dot/objectgraph-agent.git
%cd objectgraph-agent
!git tag --list prereg-v1   # must print prereg-v1
!pip -q install -r requirements.txt && bash scripts/fetch_devbench.sh

In [ ]:
# 2. Free practice run (fake model, scores will be 0)
!python scripts/run_pilot.py --project python/stocktrends --provider fake --runs 0 --cache /tmp/fakecache --out /tmp/fakeruns --results /tmp/fakeresults

In [ ]:
# 3. Check the scorer (must say usable YES)
!python scripts/check_scorer.py --projects python/stocktrends python/lice python/TextCNN

In [ ]:
# 4. Show the frozen plan (must print prereg-v1.1 and provider vercel)
%cd /content/objectgraph-agent
!git tag --list 'prereg-*'
!grep -E 'provider:|name:|date_frozen' prereg/plan.yaml

In [ ]:
# 5. API key: NEVER type it into code. Either add it in Colab's Secrets panel
#    (key icon on the left, name = the variable printed below, notebook access ON),
#    or paste it into the hidden box this cell shows.
import os, getpass, yaml
prov = yaml.safe_load(open('prereg/plan.yaml'))['model']['provider']
var = {'anthropic': 'ANTHROPIC_API_KEY', 'openrouter': 'OPENROUTER_API_KEY',
       'openai': 'OPENAI_API_KEY', 'vercel': 'AI_GATEWAY_API_KEY', 'deepseek': 'DEEPSEEK_API_KEY'}.get(prov)
print('provider:', prov, '| key variable:', var)
if var:
    try:
        from google.colab import userdata
        os.environ[var] = userdata.get(var)
        print('key loaded from Colab Secrets')
    except Exception:
        os.environ[var] = getpass.getpass(f'{prov} API key (hidden): ')
        print('key set for this session only')

In [ ]:
# 6. ONE real run (your method, G7) -> report + token count
%cd /content/objectgraph-agent
!python scripts/run_pilot.py --project python/stocktrends --conditions G7 --runs 0
import json
s = json.load(open('runs/pilot/python__stocktrends/G7/run0/summary.json'))
print(f"agents={s['k']}  tokens={s['tokens']} (input {s['input_tokens']}, output {s['output_tokens']})  messages={s['messages']}")
from IPython.display import HTML
HTML(open('runs/pilot/python__stocktrends/G7/run0/report.html').read())

In [ ]:
# 7. Full pilot for the Python projects (5 teams x 3 runs each). Run only after cell 6 looks right.
%cd /content/objectgraph-agent
for p in ['python/stocktrends', 'python/lice', 'python/TextCNN']:
    !python scripts/run_pilot.py --project {p}

In [ ]:
# 8. Verdict and download (upload pilot_results.zip to Claude afterwards)
%cd /content/objectgraph-agent
!python scripts/analyse_pilot.py
!zip -qr pilot_results.zip results runs cache prereg
from google.colab import files; files.download('pilot_results.zip')